# World Bank Data360 Harmonized Country Poverty → landing_dev (Stage 1)

Full dataset by default. Set `PILOT = True` below for a small discovery slice.

- Guide: [`world_bank_data360_harmonized_country_poverty.md`](./world_bank_data360_harmonized_country_poverty.md)
- Preferred env: `data-eng/venv`. First cell bootstraps `ingestion` if needed.
- Creds: `data-eng/ingestion/.env`

## Setup (landing_dev)

In [ ]:
import sys
from pathlib import Path


def _load_notebook_bootstrap() -> None:
    here = Path.cwd().resolve()
    candidates: list[Path] = []
    for base in [here, *here.parents]:
        candidates.extend(
            [
                base / "_bootstrap.py",
                base / "notebooks" / "_bootstrap.py",
                base / "ingestion" / "notebooks" / "_bootstrap.py",
                base / "data-eng" / "ingestion" / "notebooks" / "_bootstrap.py",
            ]
        )
    for boot in candidates:
        if boot.is_file():
            nb_dir = str(boot.parent)
            if nb_dir not in sys.path:
                sys.path.insert(0, nb_dir)
            from _bootstrap import ensure_ingestion

            print("ingestion ready", ensure_ingestion())
            return
    raise ModuleNotFoundError("Could not find notebooks/_bootstrap.py")


_load_notebook_bootstrap()

from ingestion.common.auth import ensure_auth_safe
from ingestion.common.bq import get_client
from ingestion.common.config import get_settings
from ingestion.common.load import add_metadata, load_dataframe, new_run_id

# Full dump by default. Set True only for a small discovery slice.
PILOT = False

TABLE = "world_bank_data360_harmonized_country_poverty"
SOURCE_NAME = "world_bank_data360_harmonized_country_poverty"
SOURCE_URL = "https://data360api.worldbank.org/data360/data"
DATABASE_ID = "WB_HCP"
KEY_COLS = ["database_id", "indicator", "ref_area", "sex", "age", "time_period"]
PAGE_SIZE = 1000
# Pilot-only knobs (ignored when PILOT is False)
PILOT_INDICATOR = "WB_HCP_EMP_2WAP_A"
PILOT_REF_AREA = "KEN,NGA,GHA"
PILOT_START_YEAR = 2020

settings = get_settings(cli_target="landing_dev")
ensure_auth_safe(settings)
run_id = new_run_id()
print("python", sys.executable)
print("mode", "PILOT" if PILOT else "FULL")
print("target", settings.target_dataset, "project", settings.project, "run_id", run_id)

## Fetch Data360

Full: `DATABASE_ID` + `skip`/`top` pages. Pilot: one indicator, three countries.

In [ ]:
import requests
import pandas as pd


def fetch_page(params: dict) -> tuple[list, int]:
    resp = requests.get(SOURCE_URL, params=params, timeout=120)
    resp.raise_for_status()
    payload = resp.json()
    rows = payload.get("value") or []
    return rows, int(payload.get("count") or 0)


if PILOT:
    rows, total = fetch_page(
        {
            "DATABASE_ID": DATABASE_ID,
            "INDICATOR": PILOT_INDICATOR,
            "REF_AREA": PILOT_REF_AREA,
        }
    )
    print("pilot fetch count", total, "rows", len(rows))
else:
    rows = []
    skip = 0
    total = None
    while True:
        page, total = fetch_page(
            {"DATABASE_ID": DATABASE_ID, "skip": skip, "top": PAGE_SIZE}
        )
        if not page:
            break
        rows.extend(page)
        skip += len(page)
        print(f"page skip={skip - len(page)} got={len(page)} total_so_far={len(rows)} api_count={total}")
        if total and skip >= total:
            break
        if len(page) < PAGE_SIZE:
            break
    print("full fetch rows", len(rows), "api_count", total)

raw = pd.DataFrame(rows)
raw.head()

## Normalize

In [ ]:
COL_MAP = {
    "DATABASE_ID": "database_id",
    "INDICATOR": "indicator",
    "REF_AREA": "ref_area",
    "SEX": "sex",
    "AGE": "age",
    "URBANISATION": "urbanisation",
    "TIME_PERIOD": "time_period",
    "FREQ": "freq",
    "OBS_VALUE": "obs_value",
    "UNIT_MEASURE": "unit_measure",
    "OBS_STATUS": "obs_status",
    "DATA_SOURCE": "data_source",
}

rename = {c: COL_MAP[c] for c in raw.columns if c in COL_MAP}
df = pd.DataFrame(raw.rename(columns=rename))
keep = [c for c in COL_MAP.values() if c in df.columns]
df = pd.DataFrame(df.loc[:, keep])
time_period = pd.Series(df["time_period"], index=df.index).astype(str).str.strip()
time_period_date = pd.Series(
    pd.to_datetime(time_period + "-01-01", errors="coerce"),
    index=df.index,
)
df["obs_value"] = pd.to_numeric(pd.Series(df["obs_value"], index=df.index), errors="coerce")
df["time_period"] = time_period
df["time_period_date"] = time_period_date
if PILOT:
    df = pd.DataFrame(
        df.loc[time_period_date.dt.year >= PILOT_START_YEAR].reset_index(drop=True)
    )
else:
    df = pd.DataFrame(df.reset_index(drop=True))
print("rows after normalize", len(df))
df.head()

## Preview / optional local parquet

In [ ]:
ready = add_metadata(
    pd.DataFrame(df),
    source=SOURCE_NAME,
    run_id=run_id,
    source_url=SOURCE_URL,
    key_cols=KEY_COLS,
)
settings.out_dir.mkdir(parents=True, exist_ok=True)
preview_path = settings.out_dir / f"{SOURCE_NAME}_{run_id}.parquet"
ready.to_parquet(preview_path, index=False)
print("preview", preview_path, "rows", len(ready))
ready.head()

## Load into landing_dev

`replace_partition` load job (not `insert_rows_json`). Full dump is ~616k rows — expect several minutes.

In [ ]:
client = get_client(settings)
result = load_dataframe(
    client,
    settings,
    ready,
    table=TABLE,
    mode="replace_partition",
    run_id=run_id,
)
print(result)

## After review → Stage 2 (package → landing)

Package: `sources/world_bank_data360_harmonized_country_poverty/`  
`python -m ingestion run world_bank_data360_harmonized_country_poverty` with target `landing`  
(requires `INGEST_ALLOW_PROD=1` and `CI=true`)